### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="maternal_health_risk",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5DP5D",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/maternal_health_risk/ && wget -P local-data-warehouse/maternal_health_risk/ https://archive.ics.uci.edu/static/public/863/maternal+health+risk.zip && unzip local-data-warehouse/maternal_health_risk/maternal+health+risk.zip -d local-data-warehouse/maternal_health_risk/
""",
    # References
    academic_reference_bibtex="""@inproceedings{ahmed2020review,
  title={Review and analysis of risk factor of maternal health in remote area using the Internet of Things (IoT)},
  author={Ahmed, Marzia and Kashem, Mohammod Abul and Rahman, Mostafijur and Khatun, Sabira},
  booktitle={InECCE2019: Proceedings of the 5th International Conference on Electrical, Control \& Computer Engineering, Kuantan, Pahang, Malaysia, 29th July 2019},
  pages={357--365},
  year={2020},
  organization={Springer}
}
""",
    academic_reference_bibtex_key="ahmed2020review",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- Anomaly: the data has a lot of duplicates (55%).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="RiskLevel",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="RiskLevel",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/Maternal Health Risk Data Set.csv")

cat_features = [
    "RiskLevel",
]
df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,014
Columns: 7
Use sampling: False (sample size: 1,014)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['Age', 'BS', 'SystolicBP', 'DiastolicBP', 'HeartRate', 'BodyTemp']
Rows remaining as candidates after top-6 filter: 884 (of 1,014)

#### Duplicate Report
Total duplicate rows: 562 (55.42% of dataset)
Duplicate rows ignoring target: 598 (58.97% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,29,130,70,7.7,98.0,78,mid risk
1,30,140,100,15.0,98.0,70,high risk
2,50,140,95,17.0,98.0,60,high risk
3,23,120,90,7.5,98.0,60,low risk
4,17,120,80,7.5,102.0,76,low risk


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,RiskLevel,category,0.0,0.0,3.0,"low risk, mid risk, high risk"
1,BS,float64,0.0,0.0,29.0,"7.5, 6.9, 6.8, 7.0, 7.9, 15.0, 6.1, 11.0, 7.8, 6.7"
2,BodyTemp,float64,0.0,0.0,8.0,"98.0, 101.0, 102.0, 100.0, 103.0, 99.0, 98.4, 98.6"
3,Age,int64,0.0,0.0,50.0,"23, 19, 17, 15, 35, 25, 32, 22, 50, 29"
4,SystolicBP,int64,0.0,0.0,19.0,"120, 90, 140, 100, 130, 85, 110, 76, 95, 160"
5,DiastolicBP,int64,0.0,0.0,16.0,"80, 60, 90, 70, 65, 100, 85, 75, 95, 49"
6,HeartRate,int64,0.0,0.0,16.0,"70, 76, 80, 77, 66, 60, 88, 86, 78, 75"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,1014.0,29.871795,13.474386,10.0,70.0
SystolicBP,1014.0,113.198225,18.403913,70.0,160.0
DiastolicBP,1014.0,76.460552,13.885796,49.0,100.0
BS,1014.0,8.725986,3.293532,6.0,19.0
BodyTemp,1014.0,98.665089,1.371384,98.0,103.0
HeartRate,1014.0,74.301775,8.088702,7.0,90.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column    rank                         
RiskLevel 1      low risk    406  40.04
          2      mid risk    336  33.14
          3     high risk    272  26.82

In [8]:
# Target Distribution
target_df

,count,pct
RiskLevel,,
low risk,406,40.04
mid risk,336,33.14
high risk,272,26.82


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to maternal_health_risk/019d5cd8-1d5f-758b-9fec-101f35e058e6
019d5cd8-1d5f-758b-9fec-101f35e058e6
d0cadd963f403e0abcc6b638da91720030fbba7fa58b318c539e9c37ba866fdd
